In [ ]:
# ============================================================
# Data retrieval #2 (Copernicus CDS) — ERA5-Land 2 m air temperature
# Flanders | warm season Apr–Sep 2021 | monthly-by-hour @ 09:00 UTC
# Downloads and processes ERA5 T2m into a single warm-season morning field.
# ============================================================
!pip -q install cdsapi xarray netCDF4

import cdsapi, os, zipfile, glob
import xarray as xr
from google.colab import drive, userdata

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Mini Project'
raw = os.path.join(OUT, 'era5_t2m_flanders_2021warm.nc')

# --- Credentials (single CDS token from Colab Secrets — never hardcode) ---
key = userdata.get('CDS_API_KEY')
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {key}\n")

# --- Request: monthly-averaged BY HOUR, take ONLY 09:00 UTC (matches Landsat overpass) ---
c = cdsapi.Client()
c.retrieve(
    'reanalysis-era5-single-levels-monthly-means',
    {
        'product_type': 'monthly_averaged_reanalysis_by_hour_of_day',
        'variable': '2m_temperature',
        'year': '2021',
        'month': ['04','05','06','07','08','09'],
        'time': '09:00',                     # locks the morning overpass hour
        'area': [51.55, 2.5, 50.65, 5.95],   # [N, W, S, E] Flanders
        'data_format': 'netcdf',
        'download_format': 'unarchived',     # request raw NetCDF
    },
    raw
)
print("Download complete:", raw)

# --- Safety net: some CDS downloads are ZIP even with a .nc extension ---
with open(raw, 'rb') as fh:
    magic = fh.read(2)
if magic == b'PK':
    print("File is a ZIP — extracting...")
    with zipfile.ZipFile(raw) as z:
        z.extractall(os.path.join(OUT, '_era5_extract'))
    nc_files = glob.glob(os.path.join(OUT, '_era5_extract', '*.nc'))
    print("Extracted .nc files:", nc_files)
    ds = xr.open_mfdataset(nc_files)
else:
    ds = xr.open_dataset(raw)

print(ds)

In [ ]:
# ============================================================
# Process ERA5 -> single warm-season morning T2m field (°C)
# ============================================================
import numpy as np, pandas as pd

# Average the 6 months -> one field; drop extra coordinates
t2m = ds['t2m'].mean(dim='valid_time')          # mean over Apr–Sep @ 09:00
t2m = t2m.drop_vars(['number', 'expver'], errors='ignore')
t2m_c = (t2m - 273.15).rename('T2m')            # Kelvin -> Celsius

print("T2m (°C) summary:")
print(f"  min  {float(t2m_c.min()):.2f}")
print(f"  max  {float(t2m_c.max()):.2f}")
print(f"  mean {float(t2m_c.mean()):.2f}")
print(f"  cells: {t2m_c.size}")

OUT = '/content/drive/MyDrive/Mini Project'

# Save the clean NetCDF (consumed by the main analysis notebook)
t2m_c.to_netcdf(f'{OUT}/t2m_flanders_warm2021_clean.nc')

# Also save as a point CSV (lon, lat, T2m) for easy inspection
df = t2m_c.to_dataframe().reset_index()[['longitude','latitude','T2m']].dropna()
df.to_csv(f'{OUT}/t2m_flanders_points.csv', index=False)
print("\nSample T2m points:")
print(df.head())
print("Total T2m points:", len(df))